만약 Colab environment인 경우 다음과 같은 코드 실행

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
os.chdir('/content/drive/MyDrive/2026PredictingStructuralStability/src')

Mounted at /content/drive


Local environment의 경우 cd 명령어를 사용하여 working directory 조정

최종적으로, 다음 코드 조각이 이 파일이 속한 디렉토리를 지시해야 함

In [ ]:
print(os.getcwd())

/content/drive/MyDrive/2026PredictingStructuralStability/src


In [ ]:
!pip install -r ../requirements.txt

In [ ]:
import sys
import pandas as pd
import numpy as np
import PIL
import torch
import torchvision
import sklearn
import tqdm
import timm

print("=" * 30)
print("[Info] Library Versions:")
print("=" * 30)
print(f"Python       : {sys.version.split()[0]}")
print(f"PyTorch      : {torch.__version__}")
print(f"Torchvision  : {torchvision.__version__}")
print(f"Pandas       : {pd.__version__}")
print(f"NumPy        : {np.__version__}")
print(f"Pillow (PIL) : {PIL.__version__}")
print(f"Scikit-learn : {sklearn.__version__}")
print(f"tqdm         : {tqdm.__version__}")
print(f"timm         : {timm.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
print("=" * 30)

[Info] Library Versions:
Python       : 3.12.13
PyTorch      : 2.10.0+cpu
Torchvision  : 0.25.0+cpu
Pandas       : 2.2.2
NumPy        : 2.0.2
Pillow (PIL) : 11.3.0
Scikit-learn : 1.6.1
tqdm         : 4.67.3
timm         : 1.0.25
CUDA Available: False


In [13]:
import os
import pandas as pd
import numpy as np

CFG = {
    # ----------------------------------
    # 1. Base Settings
    # ----------------------------------
    'SEED': 42,

    # ----------------------------------
    # 2. Threshold & Scaling Params
    # ----------------------------------
    'MASK_FIXED_LOW_THRESHOLD': 0.01,
    'MASK_FIXED_HIGH_THRESHOLD': 0.99,
    'MASK_FIXED_EPSLION_VALUE': 3e-4,
    'MASK_LOGIT_SCALE_LOW_THRESHOLD': 0.1,
    'MASK_LOGIT_SCALE_HIGH_THRESHOLD': 0.9,
    'MASK_LOGIT_SCALING_FACTOR': 2.5,

    # ----------------------------------
    # 3. File Names
    # ----------------------------------
    'INPUT_FILENAME': 'submission_001_convnext_tiny.csv',
    'OUTPUT_FILENAME': 'final_submission.csv',
}

def seed_everything(seed=42):
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'

seed_everything(CFG['SEED'])

# ==========================================
# 0. Directory Setup
# ==========================================
base_dir = os.getcwd()
output_dir = os.path.abspath(os.path.join(os.path.dirname(base_dir), 'outputs'))
final_submit_dir = os.path.dirname(base_dir)


# ==========================================
# 1. Data Loading
# ==========================================
df = pd.read_csv(os.path.join(output_dir, CFG['INPUT_FILENAME']))
p = df['unstable_prob'].copy()
new_p = p.copy()


# ==========================================
# 2. Postprocess
# ==========================================
# Clip extream p values
mask_fixed_low = (p < CFG['MASK_FIXED_LOW_THRESHOLD'])
mask_fixed_high = (p > CFG['MASK_FIXED_HIGH_THRESHOLD'])
new_p[mask_fixed_low] = CFG['MASK_FIXED_EPSLION_VALUE']
new_p[mask_fixed_high] = 1 - CFG['MASK_FIXED_EPSLION_VALUE']

# Scale intermidate p values
def logit_scaling(prob, temp):
    # Standard logit scaling: sigmoid(T * logit(p))
    logit = np.log(prob / (1 - prob))
    return 1 / (1 + np.exp(-temp * logit))

mask_scale = (p > CFG['MASK_LOGIT_SCALE_LOW_THRESHOLD']) & (p < CFG['MASK_LOGIT_SCALE_HIGH_THRESHOLD'])
new_p[mask_scale] = logit_scaling(p[mask_scale], CFG['MASK_LOGIT_SCALING_FACTOR'])

# ==========================================
# 3. Save postprocessed results
# ==========================================
df['unstable_prob'] = new_p
df['stable_prob'] = 1 - new_p

# Save final submission
final_submit_file = os.path.join(final_submit_dir, CFG['OUTPUT_FILENAME'])
df.to_csv(final_submit_file, index=False)
print(f"Saved to {final_submit_file}")


Saved to /content/drive/MyDrive/2026PredictingStructuralStability/final_submission.csv
